# Automatic Interpolated Dataset From SLF

Pipeline Aube pour reproduire la chaine `create_dataset` puis `create_interpolate_dataset`, mais directement dans un notebook.

Entrees attendues :
- le maillage fin `.slf`,
- le fichier `.cli` du maillage fin,
- un ou plusieurs `.res` fins,
- le maillage coarse `.slf` genere dans `automatic_coarsening_test.ipynb`.

Sorties :
- un `*_base.bin` DGL sur le maillage coarse,
- des `*_interpolated.pkl` contenant `(x, y, ts)` sur les noeuds coarse.

## 1. Imports

In [1]:
from pathlib import Path
import os
import pickle
import sys

import numpy as np
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay, cKDTree
from tqdm.auto import tqdm

import torch
import dgl


def find_project_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / 'python' / 'create_dgl_dataset.py').exists():
            return path
    raise RuntimeError('Cannot find gnn_modulus_test project root.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# create_dgl_dataset changes cwd during import; reset it just after import.
from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.create_dgl_dataset import (
    NodeType,
    add_mesh_info,
    extract_node_type,
    get_dgl_graph,
    get_dynamic_node_features,
    get_node_outputs,
    get_static_node_features,
    put_boundary_infos,
    put_boundary_infos_on_changes,
)
os.chdir(PROJECT_ROOT)

PROJECT_ROOT

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/users/m24046/m24046mrcr/gnn_modulus_test')

## 2. Parametres a remplir

In [15]:
# A remplir.
FINE_MESH_SLF = '/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/T2V6_KV4_BuseV2.geo' 
FINE_CLI = '/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/cas.conlim'  

# Option 1: liste explicite de .res.
FINE_RES_FILES = [
    '/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/08_T2V6_KV4_Busev2_Q5.res',
    '/work/m24046/m24046mrcr/Aube/10_T2V6_KV4_Busev2_Q30/10_T2V6_KV4_Busev2_Q30.res',
    '/work/m24046/m24046mrcr/Aube/12_T2V6_KV4_Busev2_Q100/12_T2V6_KV4_Busev2_Q100.res'
]

# Option 2: dossier contenant des .res. Utilise seulement si FINE_RES_FILES est vide.
FINE_RES_DIR = ''
RES_GLOB = '*.res'

# Maillage coarse genere par le premier notebook.
COARSE_MESH_SLF = '/work/m24046/m24046mrcr/Aube/automatic_coarse_mesh_T2.slf'

# Dossier de sortie du dataset interpole.
OUTPUT_DIR = "/work/m24046/m24046mrcr/Aube/interpolated_dataset"
DATASET_NAME = 'Aube_interpolated'
CHUNK_SIZE = 80

# Les coordonnees du .slf sont en simple precision. 0.5 m evite de perdre les noeuds de bord preserves.
BOUNDARY_MATCH_TOL = 0.5
FILL_NAN_WITH_NEAREST = True
OVERWRITE = True

print({
    'FINE_MESH_SLF': FINE_MESH_SLF,
    'FINE_CLI': FINE_CLI,
    'FINE_RES_FILES': FINE_RES_FILES,
    'FINE_RES_DIR': FINE_RES_DIR,
    'COARSE_MESH_SLF': str(COARSE_MESH_SLF),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'DATASET_NAME': DATASET_NAME,
    'CHUNK_SIZE': CHUNK_SIZE,
})

{'FINE_MESH_SLF': '/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/T2V6_KV4_BuseV2.geo', 'FINE_CLI': '/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/cas.conlim', 'FINE_RES_FILES': ['/work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/08_T2V6_KV4_Busev2_Q5.res', '/work/m24046/m24046mrcr/Aube/10_T2V6_KV4_Busev2_Q30/10_T2V6_KV4_Busev2_Q30.res', '/work/m24046/m24046mrcr/Aube/12_T2V6_KV4_Busev2_Q100/12_T2V6_KV4_Busev2_Q100.res'], 'FINE_RES_DIR': '', 'COARSE_MESH_SLF': '/work/m24046/m24046mrcr/Aube/automatic_coarse_mesh_T2.slf', 'OUTPUT_DIR': '/work/m24046/m24046mrcr/Aube/interpolated_dataset', 'DATASET_NAME': 'Aube_interpolated', 'CHUNK_SIZE': 80}


## 3. Utilitaires

In [16]:
def require_path(path_like, name: str) -> Path:
    if not path_like:
        raise ValueError(f'{name} is empty.')
    path = Path(path_like).expanduser()
    if not path.exists():
        raise FileNotFoundError(f'{name} does not exist: {path}')
    return path


def resolve_res_files(explicit_files, res_dir, pattern='*.res') -> list[Path]:
    if explicit_files:
        paths = [require_path(p, 'FINE_RES_FILE') for p in explicit_files]
    else:
        directory = require_path(res_dir, 'FINE_RES_DIR')
        paths = sorted(directory.glob(pattern))

    if not paths:
        raise FileNotFoundError('No .res files found.')
    return paths


def read_first_available(telemac: TelemacFile, names: list[str], timestep: int = 0, default=None):
    last_error = None
    for name in names:
        try:
            return telemac.get_data_value(name, timestep)
        except Exception as exc:
            last_error = exc
    if default is not None:
        return default
    raise KeyError(f'None of these variables were found: {names}') from last_error


def interpolate_to_points(values, interpolator_points, triangulation, target_points, nearest_tree=None):
    values = np.ascontiguousarray(values)
    interpolator = LinearNDInterpolator(triangulation, values)
    out = interpolator(target_points)

    nan_mask = np.isnan(out).any(axis=1) if out.ndim == 2 else np.isnan(out)
    if np.any(nan_mask):
        if FILL_NAN_WITH_NEAREST:
            if nearest_tree is None:
                nearest_tree = cKDTree(interpolator_points)
            _, idx = nearest_tree.query(target_points[nan_mask])
            out[nan_mask] = values[idx]
        else:
            out = np.nan_to_num(out, nan=0.0)

    return out.astype('float32')

## 4. Resolution des chemins

In [18]:
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser()

fine_mesh_path = require_path(FINE_MESH_SLF, 'FINE_MESH_SLF')
fine_cli_path = require_path(FINE_CLI, 'FINE_CLI')
coarse_mesh_path = require_path(COARSE_MESH_SLF, 'COARSE_MESH_SLF')
res_paths = resolve_res_files(FINE_RES_FILES, FINE_RES_DIR, RES_GLOB)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('fine mesh:', fine_mesh_path)
print('fine cli:', fine_cli_path)
print('coarse mesh:', coarse_mesh_path)
print('output dir:', OUTPUT_DIR)
print('res files:')
for path in res_paths:
    print(' -', path)

fine mesh: /work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/T2V6_KV4_BuseV2.geo
fine cli: /work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/cas.conlim
coarse mesh: /work/m24046/m24046mrcr/Aube/automatic_coarse_mesh_T2.slf
output dir: /work/m24046/m24046mrcr/Aube/interpolated_dataset
res files:
 - /work/m24046/m24046mrcr/Aube/08_T2V6_KV4_Busev2_Q5/08_T2V6_KV4_Busev2_Q5.res
 - /work/m24046/m24046mrcr/Aube/10_T2V6_KV4_Busev2_Q30/10_T2V6_KV4_Busev2_Q30.res
 - /work/m24046/m24046mrcr/Aube/12_T2V6_KV4_Busev2_Q100/12_T2V6_KV4_Busev2_Q100.res


## 5. Chargement des maillages

In [19]:
fine_mesh = TelemacFile(str(fine_mesh_path))
fine_with_cli = TelemacFile(str(fine_mesh_path), bnd_file=str(fine_cli_path))
coarse_mesh = TelemacFile(str(coarse_mesh_path))

X_fine, triangles_fine = add_mesh_info(fine_mesh)
X_coarse, triangles_coarse = add_mesh_info(coarse_mesh)

print('fine nodes:', len(X_fine), 'fine triangles:', len(triangles_fine))
print('coarse nodes:', len(X_coarse), 'coarse triangles:', len(triangles_coarse))

print('Building fine Delaunay...')
fine_triangulation = Delaunay(X_fine)
fine_tree = cKDTree(X_fine)
print('done')

fine nodes: 313521 fine triangles: 623905
coarse nodes: 28820 coarse triangles: 52946
Building fine Delaunay...
done


## 6. Creation du graphe coarse `*_base.bin`

In [20]:
def coarse_node_type_from_fine_boundary(
    fine_with_cli: TelemacFile,
    fine_points: np.ndarray,
    coarse_points: np.ndarray,
    tolerance: float,
) -> np.ndarray:
    fine_node_type = extract_node_type(fine_with_cli.tri, fine_with_cli.get_bnd_info()).astype('float32')
    coarse_node_type = np.zeros((len(coarse_points), NodeType.SIZE), dtype='float32')
    coarse_node_type[:, NodeType.NORMAL] = 1.0

    tree = cKDTree(fine_points)
    distances, indices = tree.query(coarse_points)
    matched = distances <= tolerance
    coarse_node_type[matched] = fine_node_type[indices[matched]]

    print('matched boundary/support nodes:', int(matched.sum()), '/', len(coarse_points))
    print('coarse node type counts:', coarse_node_type.sum(axis=0).astype(int).tolist())
    return coarse_node_type


def create_coarse_base_graph(
    coarse_mesh: TelemacFile,
    fine_with_cli: TelemacFile,
    X_fine: np.ndarray,
    X_coarse: np.ndarray,
    output_path: Path,
) -> Path:
    if output_path.exists() and not OVERWRITE:
        raise FileExistsError(f'Output already exists: {output_path}')

    node_type = coarse_node_type_from_fine_boundary(
        fine_with_cli=fine_with_cli,
        fine_points=X_fine,
        coarse_points=X_coarse,
        tolerance=BOUNDARY_MATCH_TOL,
    )

    z = read_first_available(
        coarse_mesh,
        ['FOND', 'BOTTOM'],
        timestep=0,
    ).astype('float32')
    strickler = read_first_available(
        coarse_mesh,
        ['FROTTEMENT', 'STRICKLER', 'FRICTION'],
        timestep=0,
        default=np.zeros(len(X_coarse), dtype='float32'),
    ).astype('float32')

    static = np.concatenate(
        [node_type, strickler[:, None], z[:, None]],
        axis=1,
    ).astype('float32')

    graph, edge_features = get_dgl_graph(coarse_mesh.tri)
    graph.edata['x'] = torch.as_tensor(edge_features, dtype=torch.float32)
    graph.ndata['static'] = torch.as_tensor(static, dtype=torch.float32)

    dgl.save_graphs(str(output_path), [graph])
    return output_path


base_bin_path = OUTPUT_DIR / f'{DATASET_NAME}_base.bin'
create_coarse_base_graph(
    coarse_mesh=coarse_mesh,
    fine_with_cli=fine_with_cli,
    X_fine=X_fine,
    X_coarse=X_coarse,
    output_path=base_bin_path,
)

print('base graph written:', base_bin_path)

matched boundary/support nodes: 22329 / 28820
coarse node type counts: [25629, 100, 10, 3081]
base graph written: /work/m24046/m24046mrcr/Aube/interpolated_dataset/Aube_interpolated_base.bin


## 7. Generation des chunks dynamiques interpoles

In [ ]:
def write_interpolated_chunks_for_res(
    res_path: Path,
    traj_index: int,
    fine_mesh: TelemacFile,
    fine_cli_path: Path,
    X_fine: np.ndarray,
    X_coarse: np.ndarray,
    triangulation: Delaunay,
    fine_tree: cKDTree,
    output_dir: Path,
) -> list[Path]:
    res = TelemacFile(str(res_path), bnd_file=str(fine_cli_path))
    fine_static = get_static_node_features(res, fine_mesh)
    n_times = int(res.times.shape[0])

    written = []
    for start_ts in tqdm(range(0, n_times - 1, CHUNK_SIZE), desc=res_path.name):
        end_ts = min(start_ts + CHUNK_SIZE, n_times - 1)
        dynamic_data = []

        for ts in range(start_ts, end_ts):
            x_fine = get_dynamic_node_features(res, ts)
            x_future_fine = get_dynamic_node_features(res, ts + 1)

            x_fine = put_boundary_infos(x_fine, x_future_fine, fine_static)
            y_fine = get_node_outputs(x_fine, x_future_fine)
            y_fine = put_boundary_infos_on_changes(y_fine, fine_static)

            x_coarse = interpolate_to_points(
                values=x_fine,
                interpolator_points=X_fine,
                triangulation=triangulation,
                target_points=X_coarse,
                nearest_tree=fine_tree,
            )
            y_coarse = interpolate_to_points(
                values=y_fine,
                interpolator_points=X_fine,
                triangulation=triangulation,
                target_points=X_coarse,
                nearest_tree=fine_tree,
            )

            dynamic_data.append((x_coarse, y_coarse, int(ts)))

        output_path = output_dir / f'{DATASET_NAME}_{traj_index}_{start_ts}-{end_ts}_interpolated.pkl'
        if output_path.exists() and not OVERWRITE:
            raise FileExistsError(f'Output already exists: {output_path}')
        with open(output_path, 'wb') as f:
            pickle.dump(dynamic_data, f)
        written.append(output_path)

    res.close()
    return written


all_dynamic_files = []
for traj_index, res_path in enumerate(res_paths):
    files = write_interpolated_chunks_for_res(
        res_path=res_path,
        traj_index=traj_index,
        fine_mesh=fine_mesh,
        fine_cli_path=fine_cli_path,
        X_fine=X_fine,
        X_coarse=X_coarse,
        triangulation=fine_triangulation,
        fine_tree=fine_tree,
        output_dir=OUTPUT_DIR,
    )
    all_dynamic_files.extend(files)

print('written dynamic files:', len(all_dynamic_files))
for path in all_dynamic_files[:10]:
    print(' -', path)
if len(all_dynamic_files) > 10:
    print(' ...')

08_T2V6_KV4_Busev2_Q5.res: 100%|██████████| 2/2 [16:34<00:00, 497.05s/it]


10_T2V6_KV4_Busev2_Q30.res: 100%|██████████| 2/2 [16:35<00:00, 497.56s/it]


12_T2V6_KV4_Busev2_Q100.res:   0%|          | 0/2 [00:00<?, ?it/s]

## 8. Verification rapide

In [ ]:
graphs, _ = dgl.load_graphs(str(base_bin_path))
graph = graphs[0]

print('graph nodes:', graph.num_nodes())
print('graph edges:', graph.num_edges())
print('static shape:', tuple(graph.ndata['static'].shape))
print('edge shape:', tuple(graph.edata['x'].shape))

if all_dynamic_files:
    with open(all_dynamic_files[0], 'rb') as f:
        sample_chunk = pickle.load(f)
    x0, y0, ts0 = sample_chunk[0]
    print('first dynamic file:', all_dynamic_files[0])
    print('samples in chunk:', len(sample_chunk))
    print('x shape:', x0.shape)
    print('y shape:', y0.shape)
    print('ts:', ts0)
    assert x0.shape[0] == graph.num_nodes()
    assert y0.shape[0] == graph.num_nodes()
    print('shape check: ok')

## 9. Chemins pour l'entrainement

In [ ]:
print('data_dir =', base_bin_path)
print('dynamic_dir = [')
for path in all_dynamic_files:
    print(f"    '{path}',")
print(']')